# GyroSwin-FID across multiple latent sources (ID + OOD)

Compute pairwise FID heatmaps (GT-vs-GT and GT-vs-Diff) and a global-FID
table where each row is a (GyroSwin latent source, split) combination.
Splits follow `notebooks/neurips_generate_table1.ipynb`: ID = held-out
in-distribution trajectories, OOD = out-of-distribution trajectories.

Latent sources we evaluate:
* `bottleneck`     — df_unet encoder middle activations (existing default).
* `flux_head (all)`/ levels — pre-MLP physics-targeted features.
* `decoder L=-1` / `L=-2` — df_unet up-block activations 1–2 levels before
  the final 5D output (analogue of InceptionV3 pool3 in classic FID).

Real samples come from the diffusion runner's validation trajectories
(ID + OOD); matched diffusion samples are generated via `runner.sample` with
the same per-trajectory conditioning. Every latent source re-uses the same
df batches so the comparison is apples-to-apples.

Heavy lifting lives in [neurips_fid_gyroswin_latents.py](neurips_fid_gyroswin_latents.py).

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
PROJECT_ROOT = "/system/user/galletti/git/neural-gyrokinetics-gitlab"
for _p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import torch
from notebooks.neurips_fid_gyroswin_latents import (
    LatentSource, setup, collect_real, generate_diff,
    split_by_traj_set, extract_features, compute_fid_set,
    plot_heatmap_grid, build_table,
)
from notebooks.neurips_generate_table1 import (
    TRAJECTORIES_ID, TRAJECTORIES_OOD, free_cuda,
)

torch.use_deterministic_algorithms(False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")

In [ ]:
# ---- paths (same as notebooks/neurips_generate_table1.ipynb) ---------
CKPT_ROOT           = "/restricteddata/ukaea/checkpoints/neurips26"
DIFF_CKPT_DIR       = f"{CKPT_ROOT}/DIFF_FLOW/20260412_180101_948"
AE_CHECKPOINT       = f"{CKPT_ROOT}/AE_noCond/20260405_022851_327/best.pth"
DATA_PREP           = Path("/local00/bioinf/galletti/preprocessed_kvikio")
GYROSWIN_CHECKPOINT = "/restricteddata/ukaea/checkpoints/scaling_law/gyroswin_xxl_fluxavg_cond_nodrop_l1"

# ---- knobs ----------------------------------------------------------
N_PER_TRAJ        = 32                          # snapshots per traj (real + matched diff)
GEN_BATCH_SIZE    = 32
FEAT_BATCH_SIZE   = 8                           # GyroSwin forward batch (memory)
N_DENOISING_STEPS = 15
PCA_COMPONENTS    = 64

# Latent sources to compare. Add/remove rows freely.
LATENT_SOURCES = [
    LatentSource("bottleneck",      source="bottleneck"),
    LatentSource("flux_head (all)", source="flux_head"),
    LatentSource("flux_head L0",    source="flux_head", level=0),  # bottleneck-mix
    LatentSource("flux_head L1",    source="flux_head", level=1),  # one above
    LatentSource("decoder L=-1",    source="decoder",   level=-1),  # 1 level before 5D
]

# Combined ID+OOD trajectory list -> single diff runner valset.
VALID_H5 = [
    *(t.replace("_ifft_realpotens", "") + ".h5" for t in TRAJECTORIES_ID),
    *(t.replace("_ifft_realpotens", "") + ".h5" for t in TRAJECTORIES_OOD),
]
print(f"trajectories: ID={len(TRAJECTORIES_ID)}  OOD={len(TRAJECTORIES_OOD)}  "
      f"|  latent sources={len(LATENT_SOURCES)}")

## Build models

In [ ]:
runner, gs_model, gs_cfg, gs_cond_keys = setup(
    DIFF_CKPT_DIR, AE_CHECKPOINT, DATA_PREP,
    GYROSWIN_CHECKPOINT, valid_traj_h5_names=VALID_H5, device=DEVICE,
)
print(f"GyroSwin cond_keys: {gs_cond_keys}")
print(f"diff cond_keys:     {sorted(runner.cfg.model.conditioning)}")
print(f"# up_blocks (decoder depth): {len(gs_model.df_unet.up_blocks)}")

## GyroSwin sanity check

Forward one validation sample through GyroSwin, render the 5D reconstruction,
and report the relative L2 error. Catches the two ways this pipeline silently
breaks: weights that didn't actually load (rel L2 ≈ 1) and an input-normalization
mismatch between the diff valset and what GyroSwin expects.

Then dry-run each `LATENT_SOURCES` entry on a 2-sample mini-batch to make sure
the hooks fire and the features have non-degenerate shape/stats.

In [ ]:
from notebooks.neurips_fid_gyroswin_latents import (
    gyroswin_recon_sanity, latent_extraction_sanity,
)

_ = gyroswin_recon_sanity(gs_model, runner, gs_cond_keys, DEVICE)
latent_extraction_sanity(gs_model, runner, gs_cond_keys, LATENT_SOURCES, DEVICE)

## Collect real validation snapshots and matched diffusion samples

One pass over the (ID + OOD) valset; we then split the result by
trajectory set to recover ID-only and OOD-only views.

In [ ]:
real_by_fi = collect_real(runner, n_per_traj=N_PER_TRAJ)
gen_by_fi  = generate_diff(runner, real_by_fi,
                           n_denoising_steps=N_DENOISING_STEPS,
                           batch_size=GEN_BATCH_SIZE)
free_cuda()

real_split = split_by_traj_set(real_by_fi, runner, TRAJECTORIES_ID, TRAJECTORIES_OOD)
gen_split  = split_by_traj_set(gen_by_fi,  runner, TRAJECTORIES_ID, TRAJECTORIES_OOD)

## Extract features and compute FID for each (latent source, split)

In [ ]:
results = {}
for ls in LATENT_SOURCES:
    for split in ("ID", "OOD"):
        if not real_split[split]:
            continue
        print(f"\n--- {ls.name}  ({split})  source={ls.source!r}  level={ls.level} ---")
        real_feats = extract_features(
            gs_model, real_split[split], runner, gs_cond_keys, ls,
            batch_size=FEAT_BATCH_SIZE, device=DEVICE, desc=f"real:{ls.name}/{split}",
        )
        gen_feats = extract_features(
            gs_model, gen_split[split], runner, gs_cond_keys, ls,
            batch_size=FEAT_BATCH_SIZE, device=DEVICE, desc=f"gen:{ls.name}/{split}",
        )
        results[(ls.name, split)] = compute_fid_set(
            real_feats, gen_feats, n_components=PCA_COMPONENTS,
        )
        free_cuda()
        r = results[(ls.name, split)]
        print(f"  global FID = {r['fid_global']:.3f}  "
              f"(raw dim {r['raw_dim']} -> pca {r['pca_dim']}, "
              f"{r['explained_var']:.0%} var)")

## Heatmaps — ID

In [ ]:
_id = {k: v for k, v in results.items() if k[1] == "ID"}
_ = plot_heatmap_grid(_id, title="ID — held-out in-distribution trajectories")

## Heatmaps — OOD

In [ ]:
_ood = {k: v for k, v in results.items() if k[1] == "OOD"}
_ = plot_heatmap_grid(_ood, title="OOD — out-of-distribution trajectories")

## Summary table

* `global_FID`              — pooled real vs pooled gen, on the shared PCA basis.
* `diag_FID(GT_i,Diff_i)`   — mean of per-traj FID where conditioning matches.
* `off_FID(GT_i,Diff_j)`    — mean of off-diagonal entries (pred vs wrong-traj GT).
* `GT-GT off-diag`          — reference scale: how distinguishable trajectories are in this latent space.

In [ ]:
summary = build_table(results)
summary_sorted = summary.sort_index(level=["split", "latent"])
print(summary_sorted.to_string(float_format=lambda x: f"{x:.4g}"))
summary_sorted